**ENTENDIENDO LOS DATOS**

**TRATAMIENTO DE LAS ENCUESTAS ENAHO (MODULO 1).**

In [55]:
import os
import glob
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURACIÓN DE RUTAS Y CONSTANTES
# ==============================================================================
RUTA_MODULO1 = r"D:/FUND. CIENCIA DE DATOS/CDD-2025" 
RUTA_MODULO2_2024 = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/2024"
RUTA_SALIDA = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados"

os.makedirs(RUTA_SALIDA, exist_ok=True)

# Variables a filtrar en el Módulo 1 (Carátula y Ficha de la Vivienda)
VARIABLES_A_FILTRAR = ['conglome', 'vivienda', 'hogar', 'ubigeo', 'dominio', 'estrato', 'result']

**CONFIGURACIÓN GLOBAL DEL ENTORNO**

Lee archivos CSV del INEI probando codificaciones y delimitadores ( ' ; ' , ' , ' , ' \t ').

In [56]:
def cargar_csv(ruta_archivo):
    
    encodings = ['latin1', 'cp1252', 'iso-8859-1', 'utf-8']
    separadores = [';', ',', '\t']
    
    errores = []
    
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(
                    ruta_archivo, 
                    encoding=enc, 
                    sep=sep, 
                    low_memory=False, 
                    on_bad_lines='skip'
                )
                
                # Descartar si toda la fila se leyó en una sola columna por mal delimitador
                if len(df.columns) <= 1:
                    continue
                
                df.columns = df.columns.str.lower().str.strip()
                return df
            except Exception as e:
                errores.append(f"Encoding={enc}, Sep='{sep}' -> Error: {str(e)}")
                continue
                
    detalle_error = "\n".join(errores[:3])
    raise ValueError(f"No se pudo leer el archivo: {ruta_archivo}\nDetalles:\n{detalle_error}")

**1: Alteración del Problema por Valores Faltantes en result**

Cuando la variable **result** adopta valores distintos de 1 (Completa) o 2 (Incompleta) —por ejemplo rechazo, ausente o vivienda desocupada— el encuestador no logra aplicar los módulos sociodemográficos posteriores. **Esto genera una condición de datos faltantes estructurales**.

**Impacto en la modelación:** No es posible usar características socioeconómicas (ingresos, educación, empleo) como variables predictoras de la No Respuesta, ya que estarán nulas en todos los casos donde el evento ocurra.

**Variables utilizables:** Únicamente se pueden considerar variables recolectadas al inicio de la visita o de orden geográfico/espacial (ubigeo, dominio, estrato, conglome, vivienda, hogar.).

**2: Evaluación de Utilidad del Módulo 2**

El Módulo 2 registra las características de los miembros del hogar. Al surgir una situación de rechazo o vivienda desocupada (result >= 3), no existe registro de personas en este módulo.

Lo que nos lleva a concluir que No es posible utilizar el Módulo 2 para la predicción de la no respuesta, ya que combinar (merge) el Módulo 1 con el Módulo 2 eliminaría las observaciones de no respuesta o dejaría registros totalmente vacíos en los predictores.

**3:FILTRADO DE VARIABLES**

In [57]:
# ==============================================================================
# PUNTO 3: FILTRADO DE VARIABLES VÁLIDAS
# ==============================================================================
archivos_m1 = glob.glob(os.path.join(RUTA_MODULO1, "*.csv"))

dict_dataframes_filtrados = {}
reporte_procesamiento = []

print("Procesando la recolección y filtrado de microdatos (Módulo 1)...\n")

for archivo in archivos_m1:
    nombre_archivo = os.path.basename(archivo)
    
    # Extraer el año correspondiente
    anio = next((y for y in range(2019, 2026) if str(y) in nombre_archivo), None)
    if anio is None:
        continue
    
    try:
        df = cargar_csv(archivo)
    except Exception as err:
        reporte_procesamiento.append({
            "Año": anio,
            "Registros (N)": 0,
            "Num_Cols": 0,
            "Estado": f"Error de lectura"
        })
        continue

    # Identificar columna target
    col_result = [c for c in df.columns if 'result' in c]
    if not col_result:
        reporte_procesamiento.append({
            "Año": anio,
            "Registros (N)": len(df),
            "Num_Cols": 0,
            "Estado": "Sin columna result"
        })
        continue
    target_var = col_result[0]

    # Construir lista de variables presentes en el dataset
    vars_disponibles = [v for v in VARIABLES_A_FILTRAR if v in df.columns]
    if target_var not in vars_disponibles:
        vars_disponibles.append(target_var)
        
    df_filtrado = df[vars_disponibles].copy()
    dict_dataframes_filtrados[anio] = df_filtrado
    
    # Agregar registro al reporte formal
    reporte_procesamiento.append({
        "Año": anio,
        "Registros (N)": f"{len(df_filtrado):,}",
        "N° Vars": len(df_filtrado.columns),
        "Variables Incluidas": ", ".join(df_filtrado.columns),
        "Estado": "Procesado OK"
    })

# ==============================================================================
# IMPRESIÓN DEL REPORTE TABULAR FORMAL
# ==============================================================================
df_reporte = pd.DataFrame(reporte_procesamiento).sort_values("Año")

print("=" * 85)
print("             RESUMEN : FILTRADO DE VARIABLES MÓDULO 1 (ENAHO)")
print("=" * 85)
print(df_reporte.to_string(index=False))
print("=" * 85)

Procesando la recolección y filtrado de microdatos (Módulo 1)...

             RESUMEN : FILTRADO DE VARIABLES MÓDULO 1 (ENAHO)
 Año Registros (N)  N° Vars                                         Variables Incluidas       Estado
2019        43,868        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2020        53,423        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2021        43,524        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2022        44,122        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2023        44,378        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2024        44,731        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2025        44,599        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK


**4:TABLA DE PORCENTAJES**

In [58]:

# ==============================================================================
# PUNTO 4: CUADRO DE PORCENTAJES DE NO RESPUESTA (RESULT) A NIVEL HOGAR
# ==============================================================================
print("--- Ejecutando Punto 4: Generación de Cuadro Temporal de No Respuesta ---")

if 'dict_dataframes_filtrados' not in globals() or not dict_dataframes_filtrados:
    raise NameError("El diccionario 'dict_dataframes_filtrados' no está en memoria. Ejecuta primero el Punto 3.")
 
resumen_resultados = []

for anio, df in sorted(dict_dataframes_filtrados.items()):
    # Identificar la columna 'result' (ignorando mayúsculas/minúsculas)
    cols_result = [c for c in df.columns if 'result' in c.lower()]
    
    if not cols_result:
        print(f"Advertencia: No se encontró la columna 'result' para el año {anio}")
        continue
        
    col_result = cols_result[0]
    
    # Identificar columnas para clave única del hogar
    cols_hogar = [c for c in ['conglome', 'vivienda', 'hogar'] if c in df.columns]
    
    # Desduplicar para garantizar 1 fila = 1 hogar
    if cols_hogar:
        df_hogares = df.drop_duplicates(subset=cols_hogar)
    else:
        df_hogares = df  # Si la base ya viene estructurada a nivel hogar
    
    # Cálculo porcentual del 'result' por año
    conteo = df_hogares[col_result].value_counts(normalize=True, dropna=False) * 100
    df_pct = conteo.reset_index()
    df_pct.columns = ['categoria_result', 'porcentaje']
    df_pct['anio'] = anio
    resumen_resultados.append(df_pct)

if resumen_resultados:
    df_resumen = pd.concat(resumen_resultados, ignore_index=True)
    
    # Pivotear matriz temporal: Categorías vs Años
    cuadro_tiempo = df_resumen.pivot(
        index='categoria_result', 
        columns='anio', 
        values='porcentaje'
    ).fillna(0)
    
    # Guardar matriz consolidada
    ruta_cuadro_csv = os.path.join(RUTA_SALIDA, "porcentaje_no_respuesta_2019_2025.csv")
    cuadro_tiempo.to_csv(ruta_cuadro_csv, encoding='utf-8-sig')
    
    print("\nCUADRO PORCENTUAL DE RESULTADOS DE ENTREVISTA ('RESULT') 2019 - 2025:")
    print(cuadro_tiempo.round(2))
    print(f"\nMatriz temporal guardada en: {ruta_cuadro_csv}")

# ==============================================================================
# PUNTO 5: GUARDAR DATAFRAMES FILTRADOS POR CADA AÑO
# ==============================================================================
print("\n--- Ejecutando Punto 5: Exportación de DataFrames Filtrados ---")

# Crear subcarpeta específica para mantener el orden (opcional)
carpeta_filtrados = os.path.join(RUTA_SALIDA, "dataframes_filtrados")
os.makedirs(carpeta_filtrados, exist_ok=True)

for anio, df in sorted(dict_dataframes_filtrados.items()):
    nombre_archivo = f"modulo1_filtrado_{anio}.csv"
    ruta_archivo = os.path.join(carpeta_filtrados, nombre_archivo)
    
    # Exportar individualmente
    df.to_csv(ruta_archivo, index=False, encoding='utf-8-sig')
    print(f"Guardado año {anio}: {ruta_archivo} ({len(df):,} filas)")

print("\n¡Proceso completado exitosamente!")

--- Ejecutando Punto 4: Generación de Cuadro Temporal de No Respuesta ---

CUADRO PORCENTUAL DE RESULTADOS DE ENTREVISTA ('RESULT') 2019 - 2025:
anio               2019   2020   2021   2022   2023   2024   2025
categoria_result                                                 
1                 66.06  58.38  69.17  66.21  64.85  61.75  61.12
2                 12.73   6.18   9.51  11.33  11.51  13.57  14.44
3                  3.27   2.44   3.13   3.07   3.59   3.66   3.56
4                  0.52   0.83   0.82   0.80   0.60   0.55   0.66
5                  5.50   3.47   6.19   6.42   6.58   7.02   6.82
7                 11.92  28.70  11.18  12.17  12.87  13.45  13.40

Matriz temporal guardada en: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados\porcentaje_no_respuesta_2019_2025.csv

--- Ejecutando Punto 5: Exportación de DataFrames Filtrados ---
Guardado año 2019: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados\dataframes_filtrados\modulo1_filtrado_2019.csv (43,868 filas)

**TRANSFORMANDO LOS DATOS**

In [2]:
import os
import glob
import pandas as pd
import numpy as np

**RUTAS**

In [3]:
# Configuración de Rutas de Trabajo
DIR_INPUT = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_filtrados"
DIR_OUTPUT = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_transformados"

os.makedirs(DIR_OUTPUT, exist_ok=True)

# Carga de archivos de los años 2024 y 2025
archivos_m1 = glob.glob(os.path.join(DIR_INPUT, "*.csv"))
dict_dfs = {}

for ruta in [f for f in archivos_m1 if '2024' in f or '2025' in f]:
    anio = '2024' if '2024' in ruta else '2025'
    df = pd.read_csv(ruta, low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
    dict_dfs[anio] = df

print("DataFrames 2024 y 2025 cargados en memoria exitosamente.")

DataFrames 2024 y 2025 cargados en memoria exitosamente.


**1. CREACION DE LA VARIABLE DICOTOMICA target**

In [ ]:
# ==============================================================================
# PUNTO 1: CREACIÓN DE LA VARIABLE TARGET
# ==============================================================================
for anio, df in dict_dfs.items():
    col_result = [c for c in df.columns if 'result' in c][0]
    df['target'] = np.where(df[col_result].isin([1, 2]), 0, 1)

# Reporte en Tabla Formal
resumen_target = []
for anio, df in sorted(dict_dfs.items()):
    conteo = df['target'].value_counts(normalize=True) * 100
    resumen_target.append({
        "Año": anio,
        "Total Hogares (N)": f"{len(df):,}",
        "Respuesta - 0 (%)": f"{conteo.get(0, 0):.2f}%",
        "No Respuesta - 1 (%)": f"{conteo.get(1, 0):.2f}%"
    })

df_tabla_p1 = pd.DataFrame(resumen_target)
print("=" * 70)
print("             DISTRIBUCIÓN DE LA VARIABLE TARGET")
print("=" * 70)
print(df_tabla_p1.to_string(index=False))
print("=" * 70)

             DISTRIBUCIÓN DE LA VARIABLE TARGET (PUNTO 1)
 Año Total Hogares (N) Respuesta - 0 (%) No Respuesta - 1 (%)
2024            44,731            75.32%               24.68%
2025            44,599            75.57%               24.43%


**2. EXTRACCION DE DEPARTAMENTO Y PROVINCIA DESDE (ubigeo)**

In [ ]:
for anio, df in dict_dfs.items():
    if 'ubigeo' in df.columns:
        ubigeo_str = df['ubigeo'].astype(str).str.zfill(6)
        df['departamento_cod'] = ubigeo_str.str[:2]
        df['provincia_cod'] = ubigeo_str.str[:4]

# Reporte de Cobertura
resumen_ubigeo = []
for anio, df in sorted(dict_dfs.items()):
    resumen_ubigeo.append({
        "Año": anio,
        "Departamentos Unicos": df['departamento_cod'].nunique(),
        "Provincias Unicas": df['provincia_cod'].nunique(),
        "Valores Nulos (%)": f"{df['departamento_cod'].isnull().mean() * 100:.2f}%"
    })

df_tabla_p2 = pd.DataFrame(resumen_ubigeo)
print("=" * 70)
print("             EXTRACCIÓN GEOGRÁFICA UBIGEO")
print("=" * 70)
print(df_tabla_p2.to_string(index=False))
print("=" * 70)

             EXTRACCIÓN GEOGRÁFICA UBIGEO (PUNTO 2)
 Año  Departamentos Unicos  Provincias Unicas Valores Nulos (%)
2024                    25                196             0.00%
2025                    25                196             0.00%


**3. CLASIFICACIÓN DEL AREA RESIDENCIAL (area_urbano_rural) a partir de (estrato)**

In [ ]:
for anio, df in dict_dfs.items():
    if 'estrato' in df.columns:
        df['estrato_num'] = pd.to_numeric(df['estrato'], errors='coerce')
        df['area_urbano_rural'] = np.where(df['estrato_num'] <= 5, 'Urbano', 'Rural')
        df['es_urbano'] = np.where(df['estrato_num'] <= 5, 1, 0)

# Reporte Formal Urbano/Rural
resumen_area = []
for anio, df in sorted(dict_dfs.items()):
    dist = df['area_urbano_rural'].value_counts(normalize=True) * 100
    resumen_area.append({
        "Año": anio,
        "Área Urbana (%)": f"{dist.get('Urbano', 0):.2f}%",
        "Área Rural (%)": f"{dist.get('Rural', 0):.2f}%"
    })

df_tabla_p3 = pd.DataFrame(resumen_area)
print("=" * 70)
print("             CLASIFICACIÓN DE ÁREA RESIDENCIAL")
print("=" * 70)
print(df_tabla_p3.to_string(index=False))
print("=" * 70)

             CLASIFICACIÓN DE ÁREA RESIDENCIAL (PUNTO 3)
 Año Área Urbana (%) Área Rural (%)
2024          65.99%         34.01%
2025          66.06%         33.94%


**4. MAPEO DE REGIÓN NATURAL (region_natural)**

In [8]:
for anio, df in dict_dfs.items():
    if 'dominio' in df.columns:
        df['dominio_num'] = pd.to_numeric(df['dominio'], errors='coerce')
        
        condiciones = [
            df['dominio_num'] == 8,
            df['dominio_num'].isin([1, 2, 3]),
            df['dominio_num'].isin([4, 5, 6]),
            df['dominio_num'] == 7
        ]
        etiquetas = ['Lima Metropolitana', 'Costa', 'Sierra', 'Selva']
        df['region_natural'] = np.select(condiciones, etiquetas, default='Sin Clasificar')

# Reporte de distribución por Región Natural
resumen_region = []
for anio, df in sorted(dict_dfs.items()):
    dist = df['region_natural'].value_counts(normalize=True) * 100
    resumen_region.append({
        "Año": anio,
        "Lima Metro (%)": f"{dist.get('Lima Metropolitana', 0):.2f}%",
        "Costa (%)": f"{dist.get('Costa', 0):.2f}%",
        "Sierra (%)": f"{dist.get('Sierra', 0):.2f}%",
        "Selva (%)": f"{dist.get('Selva', 0):.2f}%"
    })

df_tabla_p4 = pd.DataFrame(resumen_region)
print("=" * 75)
print("             DISTRIBUCIÓN POR REGIÓN NATURAL")
print("=" * 75)
print(df_tabla_p4.to_string(index=False))
print("=" * 75)

             DISTRIBUCIÓN POR REGIÓN NATURAL
 Año Lima Metro (%) Costa (%) Sierra (%) Selva (%)
2024         12.46%    29.61%     37.27%    20.65%
2025         12.50%    29.86%     37.12%    20.52%


**5. VARIABLES DISCRETAS Y CODIFICACIÓN DE VARIABLES**

In [9]:
for anio, df in dict_dfs.items():
    # Trimestre a partir del mes
    if 'mes' in df.columns:
        df['mes_num'] = pd.to_numeric(df['mes'], errors='coerce')
        df['trimestre'] = np.ceil(df['mes_num'] / 3).astype('Int64')

    # Codificación de variables discretas/factores
    cols_a_codificar = ['departamento_cod', 'provincia_cod', 'area_urbano_rural', 'region_natural', 'dominio']
    for col in cols_a_codificar:
        if col in df.columns:
            df[f'{col}_code'] = df[col].astype('category').cat.codes

print("Variables numéricas, temporales y códigos numéricos generados.")

Variables numéricas, temporales y códigos numéricos generados.


**6. ESTADISTICO AGREGADO COMPARATIVO**

In [10]:
list_reportes = []

for anio, df in sorted(dict_dfs.items()):
    dimensiones = [
        ('area_urbano_rural', 'Área Residencial'),
        ('region_natural', 'Región Natural'),
        ('trimestre', 'Trimestre')
    ]
    
    for col, dim_nombre in dimensiones:
        if col in df.columns:
            tabla = df.groupby(col).agg(
                Total_Hogares=('target', 'count'),
                No_Respuesta=('target', 'sum'),
                Tasa_No_Respuesta=('target', lambda x: x.mean() * 100)
            ).reset_index()
            tabla.columns = ['Categoria', 'Total_Hogares', 'No_Respuesta', 'Tasa_No_Respuesta_%']
            tabla['Dimension'] = dim_nombre
            tabla['Año'] = anio
            list_reportes.append(tabla)

df_stats_consolidado = pd.concat(list_reportes, ignore_index=True)

# Presentación pivoteada para informe
tabla_comparativa = df_stats_consolidado.pivot(
    index=['Dimension', 'Categoria'],
    columns='Año',
    values='Tasa_No_Respuesta_%'
).reset_index()

print("=" * 80)
print("      TABLA COMPARATIVA: TASA DE NO RESPUESTA (%) POR DIMENSIÓN (2024 vs 2025)")
print("=" * 80)
print(tabla_comparativa.round(2).to_string(index=False))
print("=" * 80)

      TABLA COMPARATIVA: TASA DE NO RESPUESTA (%) POR DIMENSIÓN (2024 vs 2025)
       Dimension          Categoria  2024  2025
  Región Natural              Costa 22.09 21.90
  Región Natural Lima Metropolitana 26.60 25.96
  Región Natural              Selva 21.95 21.36
  Región Natural             Sierra 27.61 27.66
Área Residencial              Rural 23.37 23.29
Área Residencial             Urbano 25.36 25.02


**7. EXPORTACIÓN DE DATAFRAMES TRANSFORMADOS**

In [11]:
for anio, df in dict_dfs.items():
    ruta_salida_csv = os.path.join(DIR_OUTPUT, f"enaho_transformado_{anio}.csv")
    df.to_csv(ruta_salida_csv, index=False, encoding='utf-8-sig')
    print(f"DataFrame transformado {anio} guardado en: {ruta_salida_csv}")

# Exportar reporte estadístico
ruta_stats = os.path.join(DIR_OUTPUT, "reporte_estadistico_no_respuesta_2024_2025.csv")
df_stats_consolidado.to_csv(ruta_stats, index=False, encoding='utf-8-sig')
print(f"Reporte estadístico comparativo guardado en: {ruta_stats}")

DataFrame transformado 2025 guardado en: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_transformados\enaho_transformado_2025.csv
DataFrame transformado 2024 guardado en: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_transformados\enaho_transformado_2024.csv
Reporte estadístico comparativo guardado en: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_transformados\reporte_estadistico_no_respuesta_2024_2025.csv
